## Biohub — Cell Tracking During Development

End-to-end pipeline: read each test `.zarr` volume → detect & segment nuclei
→ track them across time → write `submission.csv` in the exact schema the
competition expects. Two segmentation+tracking backends are included:

- **`process_dataset`** — classical multi-scale LoG detection + 3D watershed
  for segmentation, [`hoct`](https://github.com/royerlab/hoct)
  (`general_v0`) for tracking. Configured with `hoct`'s own defaults and the
  dataset's real (anisotropic) voxel spacing.
- **`process_dataset_ultrack`** — [`ultrack`](https://github.com/royerlab/ultrack)
  end-to-end: multi-hypotheses segmentation solved jointly with tracking via
  an ILP, the same method royerlab's own group uses for zebrafish light-sheet
  nuclei tracking. Optional — set `USE_ULTRACK_FOR_SUBMISSION = True` below
  to use it for the actual submission once you've confirmed it scores higher
  for your data (see the comparison methodology in the dev notebook).

Run all cells top to bottom; the last cell writes and validates
`submission.csv`.

**Running with internet OFF (Kaggle scoring)**: the first code cell installs
every non-default dependency from an attached offline wheelhouse and loads
the HOCT weights from an attached local file — nothing here calls the
internet. See `prepare_offline_assets.py` (provided alongside this notebook)
for how to build and attach those two Kaggle Datasets once, beforehand.


In [1]:
# ============================================================
# OFFLINE SETUP — Kaggle, no internet
#
# Auto-discovers the wheelhouse + HOCT weights anywhere under /kaggle/input.
#
# Install strategy: force-upgrade everything needed BEFORE checking/importing
# any of it. Python caches every module the first time it's imported
# (sys.modules) — importing a package just to "check if it's available" and
# then upgrading it afterwards is a no-op for the rest of this process, since
# later `import`s just reuse the already-cached (old) module object. That's
# also why numpy itself is left alone: Kaggle's auto-generated starter cell
# already imported it before this cell ever runs, so upgrading it now would
# only corrupt the already-loaded copy without any later code ever seeing
# the new one anyway.
# ============================================================

from pathlib import Path
import importlib
import subprocess
import sys

if "numpy" in sys.modules:
    print("ℹ️  numpy is already imported in this session (normal — Kaggle's "
          "auto-generated starter cell does this). Nothing below asks pip "
          "to upgrade numpy, so this is fine.")

KAGGLE_INPUT = Path("/kaggle/input")

print("=" * 70)
print(f"Datasets attached under {KAGGLE_INPUT}:")
print("=" * 70)
if KAGGLE_INPUT.exists():
    for p in sorted(KAGGLE_INPUT.iterdir()):
        print(" -", p.name)
else:
    print(f"  ⚠️  {KAGGLE_INPUT} does not exist at all.")
print("=" * 70)

_whl_files = sorted(KAGGLE_INPUT.rglob("*.whl")) if KAGGLE_INPUT.exists() else []
WHEELHOUSE_DIR = _whl_files[0].parent if _whl_files else (KAGGLE_INPUT / "biohub-offline-wheelhouse" / "wheelhouse")
print(f"Wheelhouse resolved to: {WHEELHOUSE_DIR}  ({len(_whl_files)} .whl files found under {KAGGLE_INPUT})")

_weights_found = sorted(KAGGLE_INPUT.rglob("general_v0.pt")) if KAGGLE_INPUT.exists() else []
HOCT_WEIGHTS_PATH = _weights_found[0] if _weights_found else (KAGGLE_INPUT / "biohub-offline-wheelhouse" / "general_v0.pt")
print(f"HOCT weights resolved to: {HOCT_WEIGHTS_PATH}  (found: {bool(_weights_found)})")
print("=" * 70)

if not _whl_files:
    raise RuntimeError(
        f"No .whl files found anywhere under {KAGGLE_INPUT} — is the "
        f"wheelhouse dataset actually attached to this notebook?"
    )

# pip package name -> import name. Force-upgraded (never pre-checked/
# imported first) so tracksdata/hoct see current-enough versions of each —
# e.g. Kaggle's pre-installed polars lacks pl.Float16, which tracksdata
# needs.
FORCE_UPGRADE = {
    "zarr": "zarr",
    "geff": "geff",
    "ilpy": "ilpy",
    "gurobipy": "gurobipy",
    "spatial-graph": "spatial_graph",
    "rustworkx": "rustworkx",
    "bidict": "bidict",
    "rstar-python": "rstar_python",
    "polars": "polars",
    "pyarrow": "pyarrow",
    "sqlalchemy": "sqlalchemy",
}
# installed with --no-deps, in this order (tracksdata before hoct, since
# hoct itself imports tracksdata) — both declare a numpy/matplotlib floor
# slightly above what Kaggle ships; --no-deps trusts the already-installed,
# slightly-older versions to work in practice rather than trying (and
# failing) to upgrade numpy mid-session.
NO_DEPS_ORDER = ["tracksdata", "hoct"]
IMPORT_NAME = {"tracksdata": "tracksdata", "hoct": "hoct"}
# Only needed if you flip USE_ULTRACK_FOR_SUBMISSION = True further down.
OPTIONAL_PACKAGES = {"ultrack": "ultrack"}


def _pip_install(pip_names, extra_args):
    result = subprocess.run(
        [sys.executable, "-m", "pip", "install",
         "--no-index", "--find-links", str(WHEELHOUSE_DIR),
         *extra_args, *pip_names],
        capture_output=True, text=True, timeout=900,
    )
    if result.returncode != 0:
        print(result.stdout[-2000:])
        print(result.stderr[-2000:])
    return result.returncode == 0


print("=" * 70)
print("Installing dependencies (offline)")
print("=" * 70)

print(f"Installing (upgrade): {list(FORCE_UPGRADE)}")
_pip_install(list(FORCE_UPGRADE), ["--upgrade"])

for pip_name in NO_DEPS_ORDER:
    print(f"Installing (--no-deps): {pip_name}")
    _pip_install([pip_name], ["--no-deps", "--upgrade"])

# ---- now do the first real import of any of these in this process ----
print("=" * 70)
print("Dependency check")
print("=" * 70)

status = {}
for pip_name, import_name in {**FORCE_UPGRADE, **IMPORT_NAME}.items():
    try:
        importlib.import_module(import_name)
        status[import_name] = "available"
    except Exception as exc:
        status[import_name] = f"MISSING — import raised: {type(exc).__name__}: {exc}"
    print(f"  {import_name:<16} {status[import_name]}")

_missing_required = [k for k, v in status.items() if v.startswith("MISSING")]
if _missing_required:
    raise RuntimeError(
        f"Missing required packages: {_missing_required}. See the per-package "
        f"error printed above for the actual reason."
    )

for pip_name, import_name in OPTIONAL_PACKAGES.items():
    try:
        importlib.import_module(import_name)
        result = "available"
    except Exception:
        _pip_install([pip_name], ["--no-deps"])
        try:
            importlib.import_module(import_name)
            result = "available"
        except Exception as exc:
            result = f"MISSING ({exc})"
    print(f"  {import_name:<16} {result}  (optional — only used if "
          f"USE_ULTRACK_FOR_SUBMISSION=True below)")

if not HOCT_WEIGHTS_PATH.exists():
    raise RuntimeError(
        f"HOCT weights not found at {HOCT_WEIGHTS_PATH}. Confirm "
        f"general_v0.pt was uploaded into the attached dataset."
    )
print(f"  HOCT weights     found at {HOCT_WEIGHTS_PATH}")
print("=" * 70)

# NOTE on gurobipy: hoct's ILP solver depends on it. It installs with a free
# size-limited license active by default (no key needed) — normally enough
# because hoct tiles large candidate graphs into small per-tile problems. If
# tracking still fails with a Gurobi license error, either get a free
# academic/trial license at https://www.gurobi.com/, or set
# USE_ULTRACK_FOR_SUBMISSION = True further down to use ultrack instead
# (its default solver is open-source and needs no license at all).


ℹ️  numpy is already imported in this session (normal — Kaggle's auto-generated starter cell does this). Nothing below asks pip to upgrade numpy, so this is fine.
Datasets attached under /kaggle/input:
 - competitions
 - datasets
Wheelhouse resolved to: /kaggle/input/datasets/emankhattap/biohub-offline-wheelhouse  (207 .whl files found under /kaggle/input)
HOCT weights resolved to: /kaggle/input/datasets/emankhattap/biohub-offline-wheelhouse/general_v0.pt  (found: True)
Installing dependencies (offline)
Installing (upgrade): ['zarr', 'geff', 'ilpy', 'gurobipy', 'spatial-graph', 'rustworkx', 'bidict', 'rstar-python', 'polars', 'pyarrow', 'sqlalchemy']
Installing (--no-deps): tracksdata
Installing (--no-deps): hoct
Dependency check
  zarr             available
  geff             available
  ilpy             available
  gurobipy         available
  spatial_graph    available
  rustworkx        available
  bidict           available
  rstar_python     available
  polars           available

/usr/local/lib/python3.12/dist-packages/dask/array/image.py:7: FutureWarning: `find_available_plugins` is deprecated since version 0.25 and will be removed in version 0.27. The plugin infrastructure of `skimage.io` is deprecated. Instead, use `imageio` or other I/O packages directly.
  from skimage.io import imread as sk_imread
/usr/lib/python3.12/importlib/__init__.py:90: FutureWarning: `reset_plugins` is deprecated since version 0.25 and will be removed in version 0.27. The plugin infrastructure of `skimage.io` is deprecated. Instead, use `imageio` or other I/O packages directly.
  return _bootstrap._gcd_import(name[level:], package, level)


  tracksdata       available
  hoct             available
  ultrack          MISSING (No module named 'higra')  (optional — only used if USE_ULTRACK_FOR_SUBMISSION=True below)
  HOCT weights     found at /kaggle/input/datasets/emankhattap/biohub-offline-wheelhouse/general_v0.pt


In [2]:
# ============================================================
# Imports
# ============================================================

import gc
import os

# Reduces CUDA out-of-memory failures from fragmentation on dense datasets
# (this competition's nuclei counts can hit ~400+ per frame). Must be set
# before torch initializes its CUDA allocator, so it's here at the very top.
os.environ.setdefault("PYTORCH_CUDA_ALLOC_CONF", "expandable_segments:True")
import warnings
from pathlib import Path

import numpy as np
import pandas as pd
import zarr
import torch

from tqdm.auto import tqdm

from scipy.ndimage import (
    gaussian_filter,
    gaussian_laplace,
    gaussian_gradient_magnitude,
)
from scipy.spatial import cKDTree

from skimage import exposure
from skimage.feature import peak_local_max
from skimage.segmentation import watershed
from skimage.measure import regionprops_table
from skimage.morphology import remove_small_objects, remove_small_holes

from hoct import load_model, predict
from tracksdata.functional import TilingScheme

warnings.filterwarnings("ignore")

# ============================================================
# Competition paths
# ============================================================

# Kaggle code-competition mount point (fixed from a local dev path — this is
# where the actual competition data lives when this notebook runs on Kaggle):
COMP_DIR = Path("/kaggle/input/competitions/biohub-cell-tracking-during-development")

TRAIN_DIR = COMP_DIR / "train"
TEST_DIR = COMP_DIR / "test"

OUTPUT_DIR = Path("/kaggle/working")
OUTPUT_DIR.mkdir(exist_ok=True)

SAMPLE_SUBMISSION_PATH = COMP_DIR / "sample_submission.csv"

# ============================================================
# Physical voxel spacing (µm)
# ============================================================

VOXEL_SIZE_UM = np.array([1.625, 0.40625, 0.40625])  # (z, y, x)

# ============================================================
# Classical detection / segmentation configuration
# ============================================================

# NOTE: in the original notebook, the `background_sigma` actually used inside
# the per-frame background-subtraction loop (cells 25A/25C/25D) was NOT the
# raw voxel size — it was computed as BACKGROUND_SIGMA_UM / VOXEL_SIZE_UM
# (see the exploration cell "Cell 9: Image Preprocessing"). We reproduce that
# exact derivation here instead of hardcoding the voxel size, to avoid
# silently changing the effective background-blur radius.
BACKGROUND_SIGMA_UM = 8.0                                       # µm
BACKGROUND_SIGMA = tuple(BACKGROUND_SIGMA_UM / VOXEL_SIZE_UM)   # -> voxel-space sigma, e.g. (4.923, 19.692, 19.692)

LOG_SIGMAS = [1.5, 2.5, 3.5, 5.0]              # Multi-scale LoG sigmas (voxels)
LOG_THRESHOLD_FRACTION = 0.03                  # Adaptive threshold: fraction of the 99.9th-percentile response (was: of response.max() — see fix note above)
MASK_THRESHOLD_FRACTION = 0.08                 # Foreground-mask threshold: fraction of the 99.9th-percentile response (was: of response.max())
MIN_PEAK_DISTANCE = 3                          # Minimum distance between LoG peaks
DEDUP_RADIUS = 1.5                             # Sub-voxel duplicate-merging radius
MIN_OBJECT_SIZE = 80                           # remove_small_objects min_size
MIN_HOLE_AREA = 64                             # remove_small_holes area_threshold
GRADIENT_SIGMA = (1, 1, 1)                     # Gradient magnitude smoothing sigma
WATERSHED_COMPACTNESS = 0.001

# ============================================================
# HOCT tracking configuration
#
# FIXED: these were set well below hoct's own defaults (see the fix note
# above and https://pypi.org/project/hoct/ "Useful flags"), and HOCT_SCALE
# didn't reflect the real (anisotropic) voxel spacing at all. Both directly
# reduce the candidate edges HOCT can even consider, which shows up as lower
# edge_jaccard / division_jaccard regardless of how good the model itself is.
# ============================================================

HOCT_MODEL_NAME = "general_v0"
DEVICE = "cuda" if torch.cuda.is_available() else "cpu"

HOCT_TILE_SHAPE = (5, 32, 64, 64)         # (T, Z, Y, X) — spatial dims cut again (96->64) per user's request
HOCT_OVERLAP_SHAPE = (1, 0, 8, 8)         # (T, Z, Y, X) — scaled down to match the smaller tile

HOCT_DISTANCE_THRESHOLD = 300.0                          # was 80 — hoct's own default; safe because n_neighbors still caps candidate-graph size
HOCT_N_NEIGHBORS = 2                                      # was 3, then 5 (hoct's own default); cut to the floor per user's request
HOCT_MAX_DELTA_T = 3                                       # was 1 — hoct's own default; 1 means a single missed detection permanently breaks a track
HOCT_SCALE = (1.0, *VOXEL_SIZE_UM.tolist())                # was (1.0, 2.0, 1.0, 1.0) — now the *real* physical voxel size (t, z, y, x), matches VOXEL_SIZE_UM
HOCT_WINDOW_SIZE = 2                                        # was 3, then 5 (hoct's own default); cut to the floor per user's request — (nodes-in-window)^2 attention cost drops another ~2.25x from 3
HOCT_TEST_TIME_AUGS = 0                                     # was 4 — reverted: it was the direct cause of a CUDA OOM (each aug adds a forward pass over the same batch), and the tiling fix above already addresses the underlying node density

RANDOM_STATE = 42
np.random.seed(RANDOM_STATE)

print("=" * 60)
print("Environment ready")
print("=" * 60)
print(f"Train dir  : {TRAIN_DIR}")
print(f"Test dir   : {TEST_DIR}")
print(f"Device     : {DEVICE}")
print(f"HOCT model : {HOCT_MODEL_NAME}")
print(f"HOCT scale : {HOCT_SCALE}  (t, z, y, x — µm)")
print("=" * 60)


Environment ready
Train dir  : /kaggle/input/competitions/biohub-cell-tracking-during-development/train
Test dir   : /kaggle/input/competitions/biohub-cell-tracking-during-development/test
Device     : cuda
HOCT model : general_v0
HOCT scale : (1.0, 1.625, 0.40625, 0.40625)  (t, z, y, x — µm)


In [3]:
def process_dataset(
    dataset_path,
    hoct_model,
    *,
    voxel_size_um=VOXEL_SIZE_UM,
    background_sigma=BACKGROUND_SIGMA,
    log_sigmas=LOG_SIGMAS,
    log_threshold_fraction=LOG_THRESHOLD_FRACTION,
    mask_threshold_fraction=MASK_THRESHOLD_FRACTION,
    min_peak_distance=MIN_PEAK_DISTANCE,
    dedup_radius=DEDUP_RADIUS,
    min_object_size=MIN_OBJECT_SIZE,
    min_hole_area=MIN_HOLE_AREA,
    gradient_sigma=GRADIENT_SIGMA,
    watershed_compactness=WATERSHED_COMPACTNESS,
    tile_shape=HOCT_TILE_SHAPE,
    overlap_shape=HOCT_OVERLAP_SHAPE,
    distance_threshold=HOCT_DISTANCE_THRESHOLD,
    n_neighbors=HOCT_N_NEIGHBORS,
    max_delta_t=HOCT_MAX_DELTA_T,
    scale=HOCT_SCALE,
    window_size=HOCT_WINDOW_SIZE,
    test_time_augs=HOCT_TEST_TIME_AUGS,
    device=DEVICE,
    verbose=True,
):
    """
    ينفّذ الـ Pipeline بالكامل على dataset واحد (ملف .zarr) ويُعيد صفوف
    submission الجاهزة لهذا الـ dataset.

    الخطوات:
        1. Read dataset                (فتح الـ zarr وقراءة الفولوم 4D)
        2. Background subtraction      (لكل فريم)
        3. LoG detection                (Multi-scale Laplacian-of-Gaussian)
        4. Sub-voxel localization       (تحسين مواقع النويات)
        5. Marker construction          (لبناء watershed markers)
        6. Gradient computation         (لاستخدامه كـ image في watershed)
        7. Foreground mask              (من استجابة الـ LoG)
        8. Watershed segmentation       (3D)
        9. Region properties            (إحصائيات تشخيصية اختيارية)
       10. HOCT tracking                (بناء solution graph)
       11. Build node_rows / edge_rows

    Parameters
    ----------
    dataset_path : Path | str
        مسار ملف/فولدر الـ .zarr الخاص بالـ dataset.
    hoct_model : object
        نموذج HOCT محمّل مسبقًا (يُمرَّر بدلاً من تحميله من جديد لكل dataset).
    verbose : bool
        طباعة تقدم التنفيذ.

    Returns
    -------
    node_rows : pd.DataFrame
    edge_rows : pd.DataFrame
    """

    dataset_path = Path(dataset_path)
    dataset_name = dataset_path.stem

    # ------------------------------------------------------------
    # 1) Read dataset
    # ------------------------------------------------------------
    root = zarr.open(dataset_path, mode="r")
    image_array = root["0"]

    T = image_array.shape[0]
    volume_shape = image_array.shape[1:]

    if verbose:
        print("=" * 70)
        print(f"Processing dataset: {dataset_name}  (T={T}, shape={volume_shape})")
        print("=" * 70)

    all_centroids = []
    markers_4d = np.zeros(image_array.shape, dtype=np.int32)
    gradient_4d = np.zeros(image_array.shape, dtype=np.float32)
    mask_4d = np.zeros(image_array.shape, dtype=bool)
    labels_4d = np.zeros(image_array.shape, dtype=np.uint16)
    all_props = []
    total_dropped_markers = 0  # FIX: count instead of silently losing cells

    # ------------------------------------------------------------
    # Per-frame processing:
    # 2) Background subtraction -> 3) LoG detection ->
    # 4) Sub-voxel localization -> 5) Marker construction ->
    # 6) Gradient computation -> 7) Foreground mask ->
    # 8) Watershed segmentation -> 9) Region properties
    # ------------------------------------------------------------
    for t in tqdm(range(T), disable=not verbose, desc=f"{dataset_name}: frames"):

        # -- Load frame --
        img = image_array[t].astype(np.float32)

        # -- Normalize --
        p1, p99 = np.percentile(img, (1, 99.8))
        img = exposure.rescale_intensity(
            img, in_range=(p1, p99), out_range=(0, 1)
        ).astype(np.float32)

        # -- 2) Background subtraction --
        background = gaussian_filter(img, sigma=background_sigma)
        img_bs = img - background
        img_bs[img_bs < 0] = 0

        # -- 3) Multi-scale LoG detection --
        responses = [-gaussian_laplace(img_bs, sigma=sigma) for sigma in log_sigmas]
        scale_space = np.stack(responses)
        log_response = scale_space.max(axis=0)

        # FIX: anchor thresholds to the 99.9th percentile instead of the raw
        # max. A single hot outlier voxel (sensor noise / autofluorescent
        # speck) used to shift the *entire* frame's detection sensitivity,
        # so recall/precision swung frame-to-frame and dataset-to-dataset.
        # `log_threshold_fraction` / `mask_threshold_fraction` keep the same
        # meaning as before against this more robust anchor.
        robust_response_max = np.percentile(log_response, 99.9)

        det_thr = max(
            log_threshold_fraction * robust_response_max,
            log_response.mean() + 2.0 * log_response.std(),
        )

        coords = peak_local_max(
            log_response,
            min_distance=min_peak_distance,
            threshold_abs=det_thr,
            exclude_border=False,
        )

        # -- 4) Sub-voxel localization --
        refined = []
        for z, y, x in coords:
            if (
                z < 1 or y < 1 or x < 1
                or z >= log_response.shape[0] - 1
                or y >= log_response.shape[1] - 1
                or x >= log_response.shape[2] - 1
            ):
                continue

            patch = log_response[z - 1:z + 2, y - 1:y + 2, x - 1:x + 2]
            if patch.shape != (3, 3, 3):
                continue

            zz, yy, xx = np.mgrid[-1:2, -1:2, -1:2]
            weights = patch - patch.min()
            if weights.sum() == 0:
                continue

            dz = (zz * weights).sum() / weights.sum()
            dy = (yy * weights).sum() / weights.sum()
            dx = (xx * weights).sum() / weights.sum()

            refined.append([z + dz, y + dy, x + dx])

        refined = np.asarray(refined, dtype=np.float32)

        # -- Remove duplicated detections --
        if len(refined):
            tree = cKDTree(refined)
            visited = np.zeros(len(refined), dtype=bool)
            merged = []
            for i in range(len(refined)):
                if visited[i]:
                    continue
                neighbors = tree.query_ball_point(refined[i], r=dedup_radius)
                visited[neighbors] = True
                merged.append(refined[neighbors].mean(axis=0))
            refined = np.asarray(merged, dtype=np.float32)

        all_centroids.append(refined)

        # -- 5) Marker construction --
        markers = np.zeros(volume_shape, dtype=np.int32)
        label = 1
        for z, y, x in refined:
            zi, yi, xi = int(round(z)), int(round(y)), int(round(x))
            if not (
                0 <= zi < markers.shape[0]
                and 0 <= yi < markers.shape[1]
                and 0 <= xi < markers.shape[2]
            ):
                continue

            if markers[zi, yi, xi] == 0:
                markers[zi, yi, xi] = label
                label += 1
                continue

            # FIX: two distinct sub-voxel-refined centroids rounded to the
            # same integer voxel (rare). This used to silently drop the
            # second one entirely -> that cell never reaches labels_4d ->
            # never reaches the submission -> a guaranteed false negative.
            # Place it in the nearest free voxel of its 3x3x3 neighborhood
            # instead of dropping it.
            placed = False
            for dz in (-1, 0, 1):
                for dy in (-1, 0, 1):
                    for dx in (-1, 0, 1):
                        zzi, yyi, xxi = zi + dz, yi + dy, xi + dx
                        if (
                            0 <= zzi < markers.shape[0]
                            and 0 <= yyi < markers.shape[1]
                            and 0 <= xxi < markers.shape[2]
                            and markers[zzi, yyi, xxi] == 0
                        ):
                            markers[zzi, yyi, xxi] = label
                            label += 1
                            placed = True
                            break
                    if placed:
                        break
                if placed:
                    break
            if not placed:
                total_dropped_markers += 1
        markers_4d[t] = markers

        # -- 6) Gradient computation --
        gradient = gaussian_gradient_magnitude(img_bs, sigma=gradient_sigma).astype(np.float32)
        if gradient.max() > 0:
            gradient /= gradient.max()
        gradient_4d[t] = gradient

        # -- 7) Foreground mask (from LoG response) --
        # FIX: same robust-percentile anchor as det_thr above, instead of
        # log_response.max().
        mask = log_response > (mask_threshold_fraction * robust_response_max)
        mask = remove_small_objects(mask, min_size=min_object_size)
        mask = remove_small_holes(mask, area_threshold=min_hole_area)
        mask_4d[t] = mask

        # -- 8) Watershed segmentation --
        if markers.max() > 0 and mask.sum() > 0:
            labels = watershed(
                image=gradient,
                markers=markers,
                mask=mask,
                compactness=watershed_compactness,
            ).astype(np.uint16)
            labels_4d[t] = labels
        else:
            labels = labels_4d[t]

        # -- 9) Region properties (diagnostic) --
        props = regionprops_table(labels, properties=["label", "centroid", "area"])
        props = pd.DataFrame(props).rename(
            columns={"centroid-0": "z", "centroid-1": "y", "centroid-2": "x", "area": "volume"}
        )
        if len(props):
            props = props[["label", "z", "y", "x", "volume"]].sort_values("label").reset_index(drop=True)
        all_props.append(props)

    if verbose:
        total_nuclei = sum(len(c) for c in all_centroids)
        print(f"Detected {total_nuclei} nuclei across {T} frames "
              f"({total_nuclei / T:.1f} / frame on average)")
        if total_dropped_markers:
            print(f"⚠️  {total_dropped_markers} detection(s) had no free voxel in their "
                  f"3x3x3 neighborhood and were still dropped (very rare collision case)")

    # ------------------------------------------------------------
    # 10) HOCT tracking
    # ------------------------------------------------------------
    gc.collect()
    if torch.cuda.is_available():
        torch.cuda.empty_cache()
        torch.cuda.ipc_collect()

    tiling = TilingScheme(tile_shape=tile_shape, overlap_shape=overlap_shape)

    if verbose:
        print("Running HOCT tracking ...")

    try:
        solution_graph = predict(
            model=hoct_model,
            images=image_array,
            labels=labels_4d,
            distance_threshold=distance_threshold,
            n_neighbors=n_neighbors,
            max_delta_t=max_delta_t,
            scale=scale,
            window_size=window_size,
            tiling_scheme=tiling,
            test_time_augs=test_time_augs,
            return_solution=True,
        )
    finally:
        gc.collect()
        if torch.cuda.is_available():
            torch.cuda.empty_cache()
            torch.cuda.ipc_collect()

    solution_graph.assign_tracklet_ids()

    # ------------------------------------------------------------
    # Free the large per-frame 4D arrays now that tracking is done —
    # doesn't change results, just keeps memory usage down across
    # multiple datasets in the test loop.
    # ------------------------------------------------------------
    del labels_4d, markers_4d, gradient_4d, mask_4d
    gc.collect()
    if torch.cuda.is_available():
        torch.cuda.empty_cache()

    # ------------------------------------------------------------
    # 11) Build node_rows / edge_rows
    # ------------------------------------------------------------
    nodes = solution_graph.node_attrs(unpack=True).to_pandas()
    nodes = nodes[nodes["solution"]].copy()
    nodes["z"] = np.rint(nodes["z"]).astype(int)
    nodes["y"] = np.rint(nodes["y"]).astype(int)
    nodes["x"] = np.rint(nodes["x"]).astype(int)

    node_rows = pd.DataFrame({
        "dataset": dataset_name,
        "row_type": "node",
        "node_id": nodes["node_id"].astype(int),
        "t": nodes["t"].astype(int),
        "z": nodes["z"],
        "y": nodes["y"],
        "x": nodes["x"],
        "source_id": -1,
        "target_id": -1,
    })

    edges = solution_graph.edge_attrs(unpack=True).to_pandas()
    edges = edges[edges["solution"]].copy()

    edge_rows = pd.DataFrame({
        "dataset": dataset_name,
        "row_type": "edge",
        "node_id": -1,
        "t": -1,
        "z": -1,
        "y": -1,
        "x": -1,
        "source_id": edges["source_id"].astype(int),
        "target_id": edges["target_id"].astype(int),
    })

    if verbose:
        print(f"{dataset_name}: {len(node_rows)} node rows, {len(edge_rows)} edge rows")
        print("=" * 70)

    return node_rows, edge_rows


### Optional alternate pipeline: `ultrack`

Multi-hypotheses segmentation solved jointly with tracking via an ILP —
royerlab's own method for zebrafish light-sheet nuclei tracking. Not used by
default (see `USE_ULTRACK_FOR_SUBMISSION` below); installs quietly and skips
itself if unavailable.


In [4]:
# ============================================================
# Calibrate ultrack's segmentation_config.min_area / max_area from the
# classical LoG detector's own object sizes on one frame, instead of
# guessing — grounded in this dataset's actual nucleus sizes.
# ============================================================
from scipy.ndimage import label as _ndi_label


def estimate_area_bounds(dataset_path, voxel_size_um=VOXEL_SIZE_UM, frame_index=0):
    root = zarr.open(dataset_path, mode="r")
    image_array = root["0"]
    img = image_array[frame_index].astype(np.float32)

    p1, p99 = np.percentile(img, (1, 99.8))
    img = exposure.rescale_intensity(img, in_range=(p1, p99), out_range=(0, 1)).astype(np.float32)

    background = gaussian_filter(img, sigma=BACKGROUND_SIGMA)
    img_bs = img - background
    img_bs[img_bs < 0] = 0

    responses = [-gaussian_laplace(img_bs, sigma=s) for s in LOG_SIGMAS]
    log_response = np.stack(responses).max(axis=0)
    robust_max = np.percentile(log_response, 99.9)

    mask = log_response > (MASK_THRESHOLD_FRACTION * robust_max)
    mask = remove_small_objects(mask, min_size=MIN_OBJECT_SIZE)

    lbl, _ = _ndi_label(mask)
    areas = np.bincount(lbl.ravel())[1:]
    areas = areas[areas > 0]

    if len(areas) == 0:
        return MIN_OBJECT_SIZE, MIN_OBJECT_SIZE * 10

    min_area = max(int(np.percentile(areas, 10) * 0.5), 20)
    max_area = int(np.percentile(areas, 95) * 1.5)
    return min_area, max_area


In [5]:
# ============================================================
# Full segmentation + tracking with ultrack, in the same node_rows /
# edge_rows shape process_dataset() returns.
# ============================================================
import inspect


def process_dataset_ultrack(
    dataset_path,
    *,
    voxel_size_um=VOXEL_SIZE_UM,
    foreground_sigma_um=15.0,
    min_area=None,
    max_area=None,
    linking_max_distance_um=30.0,
    n_workers=4,
    verbose=True,
):
    """
    Segments and tracks one dataset end-to-end with ultrack:
      1. foreground  = ultrack.imgproc.detect_foreground(...)
      2. contours    = ultrack.imgproc.inverted_edt(...)      (nuclei are convex)
      3. Tracker(config).track(foreground=..., edges=contours, scale=...)
      4. tracks_df, graph = tracker.to_tracks_layer()
      5. tracks_df + graph -> node_rows / edge_rows (same schema as process_dataset())

    `graph` follows napari's Tracks-layer convention: {child_track_id:
    [parent_track_id, ...]} — used here to add the division edge from the
    parent track's last node to the child track's first node.
    """
    from ultrack import MainConfig, Tracker
    from ultrack.imgproc import detect_foreground, inverted_edt

    dataset_path = Path(dataset_path)
    dataset_name = dataset_path.stem

    root = zarr.open(dataset_path, mode="r")
    image_array = root["0"]
    # NOTE: loads the whole (T, Z, Y, X) volume into RAM, same as HOCT's own
    # `images=image_array` call already implicitly required. For datasets too
    # large for RAM, ultrack can also stream frame-by-frame — see
    # https://royerlab.github.io/ultrack/ "Out-of-memory storage".
    image_4d = np.asarray(image_array)

    if min_area is None or max_area is None:
        auto_min, auto_max = estimate_area_bounds(dataset_path, voxel_size_um)
        min_area = min_area if min_area is not None else auto_min
        max_area = max_area if max_area is not None else auto_max

    if verbose:
        print("=" * 70)
        print(f"[ultrack] {dataset_name}: min_area={min_area}, max_area={max_area}, "
              f"linking_max_distance={linking_max_distance_um}µm")

    foreground = detect_foreground(image_4d, voxel_size=voxel_size_um, sigma=foreground_sigma_um)
    contours = inverted_edt(foreground, voxel_size=voxel_size_um)

    config = MainConfig()
    config.segmentation_config.min_area = min_area
    config.segmentation_config.max_area = max_area
    config.segmentation_config.n_workers = n_workers
    config.linking_config.max_distance = linking_max_distance_um
    config.linking_config.n_workers = n_workers

    tracker = Tracker(config)

    # Different ultrack releases have named the contour-map kwarg `edges`
    # (current README) or `contours` (docs quickstart) — inspect the actual
    # installed signature instead of guessing.
    sig = inspect.signature(tracker.track)
    call_kwargs = {"foreground": foreground}
    call_kwargs["edges" if "edges" in sig.parameters else "contours"] = contours
    if "scale" in sig.parameters:
        call_kwargs["scale"] = tuple(voxel_size_um)

    tracker.track(**call_kwargs)
    tracks_df, graph = tracker.to_tracks_layer()

    tracks_df = tracks_df.sort_values(["track_id", "t"]).reset_index(drop=True)
    tracks_df["node_id"] = np.arange(len(tracks_df))

    node_rows = pd.DataFrame({
        "dataset": dataset_name,
        "row_type": "node",
        "node_id": tracks_df["node_id"].astype(int),
        "t": tracks_df["t"].astype(int),
        "z": np.rint(tracks_df["z"]).astype(int),
        "y": np.rint(tracks_df["y"]).astype(int),
        "x": np.rint(tracks_df["x"]).astype(int),
        "source_id": -1,
        "target_id": -1,
    })

    # -- consecutive-frame edges within each continuous track segment --
    edge_pairs = []
    for _, g in tracks_df.groupby("track_id"):
        g = g.sort_values("t")
        ids = g["node_id"].to_numpy()
        ts = g["t"].to_numpy()
        for i in range(len(g) - 1):
            if ts[i + 1] == ts[i] + 1:
                edge_pairs.append((int(ids[i]), int(ids[i + 1])))

    # -- division edges: parent track's last node -> child track's first node --
    last_node = tracks_df.sort_values("t").groupby("track_id")["node_id"].last().to_dict()
    first_node = tracks_df.sort_values("t").groupby("track_id")["node_id"].first().to_dict()
    for child_track_id, parent_track_ids in (graph or {}).items():
        for parent_track_id in parent_track_ids:
            if parent_track_id in last_node and child_track_id in first_node:
                edge_pairs.append((int(last_node[parent_track_id]), int(first_node[child_track_id])))

    edge_rows = pd.DataFrame(edge_pairs, columns=["source_id", "target_id"])
    edge_rows = pd.DataFrame({
        "dataset": dataset_name,
        "row_type": "edge",
        "node_id": -1,
        "t": -1,
        "z": -1,
        "y": -1,
        "x": -1,
        "source_id": edge_rows["source_id"] if len(edge_rows) else pd.Series([], dtype=int),
        "target_id": edge_rows["target_id"] if len(edge_rows) else pd.Series([], dtype=int),
    })

    if verbose:
        print(f"{dataset_name}: {len(node_rows)} node rows, {len(edge_rows)} edge rows (ultrack)")
        print("=" * 70)

    return node_rows, edge_rows


In [6]:
print("=" * 70)
print("Loading HOCT model ...")
print("=" * 70)

# Load from the local attached weights file (offline) rather than by name —
# load_model()/resolve_model() only hit the network when given a bare model
# name; a path that already exists on disk is used as-is.
hoct_model = load_model(str(HOCT_WEIGHTS_PATH), device=DEVICE)

print("Model loaded on:", DEVICE)

# ============================================================
# Which pipeline builds the submission?
# Default is False (classical LoG/watershed + HOCT, hyperparameters fixed to
# hoct's own defaults and the dataset's real voxel spacing). Flip to True to
# use ultrack (multi-hypotheses segmentation solved jointly with tracking)
# once you've confirmed on a train subset that it scores higher for you.
# ============================================================
USE_ULTRACK_FOR_SUBMISSION = False

test_datasets = sorted(TEST_DIR.glob("*.zarr"))

print(f"Found {len(test_datasets)} test dataset(s):")
for p in test_datasets:
    print(" -", p.name)

print(f"Pipeline for this submission: "
      f"{'ultrack' if USE_ULTRACK_FOR_SUBMISSION else 'classical LoG/watershed + HOCT'}")

all_nodes = []
all_edges = []

for dataset_path in test_datasets:

    print(f"Processing {dataset_path.stem}")

    if USE_ULTRACK_FOR_SUBMISSION:
        node_rows, edge_rows = process_dataset_ultrack(dataset_path)
    else:
        node_rows, edge_rows = process_dataset(dataset_path, hoct_model)

    all_nodes.append(node_rows)
    all_edges.append(edge_rows)


Loading HOCT model ...
Model loaded on: cuda
Found 4 test dataset(s):
 - 44b6_0113de3b.zarr
 - 44b6_0b24845f.zarr
 - 6bba_05b6850b.zarr
 - 6bba_05db0fb1.zarr
Pipeline for this submission: classical LoG/watershed + HOCT
Processing 44b6_0113de3b
Processing dataset: 44b6_0113de3b  (T=100, shape=(64, 256, 256))


44b6_0113de3b: frames:   0%|          | 0/100 [00:00<?, ?it/s]

Detected 42980 nuclei across 100 frames (429.8 / frame on average)
Running HOCT tracking ...


Adding edges:   0%|          | 0/100 [00:00<?, ?it/s]

[09/27/26 19:00:00] WARNING  No nodes found for time point -3                                ]8;id=899803;file:///usr/local/lib/python3.12/dist-packages/tracksdata/edges/_distance_edges.py\_distance_edges.py]8;;\:]8;id=815571;file:///usr/local/lib/python3.12/dist-packages/tracksdata/edges/_distance_edges.py#295\295]8;;\

...

                    WARNING  No nodes found for time point -1                                ]8;id=609989;file:///usr/local/lib/python3.12/dist-packages/tracksdata/edges/_distance_edges.py\_distance_edges.py]8;;\:]8;id=710276;file:///usr/local/lib/python3.12/dist-packages/tracksdata/edges/_distance_edges.py#295\295]8;;\

Adding node attributes: 100%|██████████| 100/100 [00:00<00:00, 849.43it/s]
[W927 19:47:14.655286955 init.cpp:862] Warning: Use _jit_set_fusion_strategy, bailout depth is deprecated. Setting to (STATIC, 0) (function operator())
Model inference: 624it [00:27, 22.91it/s]


[09/27/26 19:48:37] WARNING  Solver failed with Gurobi, trying Scip.                             ]8;id=338910;file:///usr/local/lib/python3.12/dist-packages/tracksdata/solvers/_ilp_solver.py\_ilp_solver.py]8;;\:]8;id=121642;file:///usr/local/lib/python3.12/dist-packages/tracksdata/solvers/_ilp_solver.py#363\363]8;;\
                             Got error:                                                                            
                             Gurobi license is not available.                                                      

[09/27/26 19:58:08] WARNING  Solver failed with Gurobi, trying Scip.                             ]8;id=743555;file:///usr/local/lib/python3.12/dist-packages/tracksdata/solvers/_ilp_solver.py\_ilp_solver.py]8;;\:]8;id=811543;file:///usr/local/lib/python3.12/dist-packages/tracksdata/solvers/_ilp_solver.py#363\363]8;;\
                             Got error:                                                                            
                             Gurobi license is not available.                                                      

6bba_05db0fb1: 90493 node rows, 86487 edge rows


In [11]:
submission = pd.concat(
    all_nodes + all_edges,
    ignore_index=True
)

submission.insert(
    0,
    "id",
    np.arange(len(submission))
)

submission_path = OUTPUT_DIR / "submission.csv"
submission.to_csv(submission_path, index=False)

print("=" * 70)
print("Submission created")
print("=" * 70)
print(f"Path      : {submission_path}")
print("Node rows :", sum(len(df) for df in all_nodes))
print("Edge rows :", sum(len(df) for df in all_edges))
print("Total rows:", len(submission))

submission.head(20)


Submission created
Path      : /kaggle/working/submission.csv
Node rows : 230805
Edge rows : 219831
Total rows: 450636


,id,dataset,row_type,node_id,t,z,y,x,source_id,target_id
0,0,44b6_0113de3b,node,0,0,22,75,156,-1,-1
1,1,44b6_0113de3b,node,1,0,58,136,199,-1,-1
2,2,44b6_0113de3b,node,2,0,58,137,204,-1,-1
3,3,44b6_0113de3b,node,3,0,60,117,221,-1,-1
4,4,44b6_0113de3b,node,4,0,17,99,87,-1,-1
5,5,44b6_0113de3b,node,5,0,45,6,190,-1,-1
6,6,44b6_0113de3b,node,6,0,12,194,61,-1,-1
7,7,44b6_0113de3b,node,7,0,26,150,94,-1,-1
8,8,44b6_0113de3b,node,8,0,17,105,84,-1,-1
9,9,44b6_0113de3b,node,9,0,55,41,217,-1,-1


In [12]:
# ============================================================
# Submission format validator
# ============================================================

EXPECTED_COLUMNS = ["id", "dataset", "row_type", "node_id", "t", "z", "y", "x",
                    "source_id", "target_id"]


def validate_submission(submission_path, test_dir=TEST_DIR, sample_submission_path=None):
    problems, warnings_ = [], []
    df = pd.read_csv(submission_path)

    # --- schema -----------------------------------------------------
    if list(df.columns) != EXPECTED_COLUMNS:
        problems.append(f"columns are {list(df.columns)}, expected {EXPECTED_COLUMNS}")

    if sample_submission_path is not None and Path(sample_submission_path).exists():
        sample_cols = list(pd.read_csv(sample_submission_path, nrows=0).columns)
        if list(df.columns) != sample_cols:
            problems.append(f"columns differ from sample_submission.csv ({sample_cols})")

    if df["id"].duplicated().any():
        problems.append("duplicate `id` values")
    if not df["id"].is_monotonic_increasing:
        warnings_.append("`id` is not sorted ascending (not required, but unusual)")

    if df.isna().any().any():
        problems.append(f"missing values in columns: {df.columns[df.isna().any()].tolist()}")

    bad_row_types = set(df["row_type"]) - {"node", "edge"}
    if bad_row_types:
        problems.append(f"unexpected row_type values: {bad_row_types}")

    # --- dataset names match the actual test set ---------------------
    test_names = {p.stem for p in sorted(Path(test_dir).glob("*.zarr"))} if Path(test_dir).exists() else None
    submitted_names = set(df["dataset"].unique())
    if test_names is not None:
        missing = test_names - submitted_names
        extra = submitted_names - test_names
        if missing:
            problems.append(f"no rows for these test datasets: {sorted(missing)}")
        if extra:
            warnings_.append(f"rows for datasets not found under {test_dir}: {sorted(extra)}")

    # --- per-dataset structural checks --------------------------------
    for ds_name, g in df.groupby("dataset"):
        nodes = g[g["row_type"] == "node"]
        edges = g[g["row_type"] == "edge"]

        if nodes["node_id"].duplicated().any():
            problems.append(f"[{ds_name}] duplicate node_id within dataset")

        node_ids = set(nodes["node_id"])
        dangling_src = set(edges["source_id"]) - node_ids
        dangling_tgt = set(edges["target_id"]) - node_ids
        if dangling_src:
            problems.append(f"[{ds_name}] {len(dangling_src)} edge source_id(s) with no matching node row")
        if dangling_tgt:
            problems.append(f"[{ds_name}] {len(dangling_tgt)} edge target_id(s) with no matching node row")

        if len(nodes) == 0:
            warnings_.append(f"[{ds_name}] has 0 predicted nodes")

    # --- report -------------------------------------------------------
    print("=" * 70)
    print(f"Validated: {submission_path}")
    print(f"Rows: {len(df):,}  |  Datasets: {df['dataset'].nunique()}  |  "
          f"Nodes: {(df['row_type'] == 'node').sum():,}  |  "
          f"Edges: {(df['row_type'] == 'edge').sum():,}")
    print("-" * 70)
    if problems:
        print(f"❌ {len(problems)} problem(s):")
        for p in problems:
            print("   -", p)
    else:
        print("✅ No structural problems found.")
    if warnings_:
        print(f"⚠️  {len(warnings_)} warning(s):")
        for w in warnings_:
            print("   -", w)
    print("=" * 70)

    return {"ok": len(problems) == 0, "problems": problems, "warnings": warnings_}


validation_result = validate_submission(
    submission_path,
    test_dir=TEST_DIR,
    sample_submission_path=globals().get("SAMPLE_SUBMISSION_PATH"),
)


Validated: /kaggle/working/submission.csv
Rows: 450,636  |  Datasets: 4  |  Nodes: 230,805  |  Edges: 219,831
----------------------------------------------------------------------
✅ No structural problems found.


In [14]:
# ============================================================
# geff ground-truth loading + T_true (estimated_number_of_nodes)
# ============================================================

def load_geff_graph(geff_path):
    """
    Read a .geff ground-truth graph as a networkx.DiGraph.
    Returns (graph, metadata_or_None) — geff.read may return either just the
    graph or a (graph, metadata) tuple depending on version.
    """
    import geff
    result = geff.read(str(geff_path), backend="networkx")
    if isinstance(result, tuple):
        graph = result[0]
        metadata = result[1] if len(result) > 1 else None
    else:
        graph, metadata = result, None
    return graph, metadata


def geff_nodes_df(graph):
    rows = [
        {"node_id": nid, "t": int(a["t"]), "z": float(a["z"]), "y": float(a["y"]), "x": float(a["x"])}
        for nid, a in graph.nodes(data=True)
    ]
    return pd.DataFrame(rows)


def geff_edges_df(graph):
    rows = [{"source_id": u, "target_id": v} for u, v in graph.edges()]
    return pd.DataFrame(rows, columns=["source_id", "target_id"])


def estimated_true_node_count(geff_path, graph=None, metadata=None):
    """
    T_true من شروط المسابقة: كل .geff فيه estimated_number_of_nodes في الـ
    metadata بتاعه — ده اللي بيخلي عقوبة adjusted-Jaccard منطقية رغم إن
    الـ ground truth مش كامل (sparse). بيجرب أكتر من مكان لإنه ممكن يختلف
    حسب نسخة geff-spec، ولو مالقاهوش بيرجع لعدد نويات الـ GT نفسه (تقدير أقل من الحقيقي).
    """
    if metadata is not None:
        for attr in ("estimated_number_of_nodes",):
            val = getattr(metadata, attr, None)
            if isinstance(val, (int, float)):
                return int(val)
        for attr in ("extra", "properties", "sparse", "model_extra"):
            val = getattr(metadata, attr, None)
            if isinstance(val, dict) and "estimated_number_of_nodes" in val:
                return int(val["estimated_number_of_nodes"])

    try:
        root = zarr.open(str(geff_path), mode="r")
        attrs = dict(root.attrs)
        candidates = [attrs, attrs.get("geff", {})]
        for c in candidates:
            if isinstance(c, dict) and "estimated_number_of_nodes" in c:
                return int(c["estimated_number_of_nodes"])
            if isinstance(c, dict):
                for v in c.values():
                    if isinstance(v, dict) and "estimated_number_of_nodes" in v:
                        return int(v["estimated_number_of_nodes"])
    except Exception as exc:
        print(f"⚠️  could not read zarr attrs for {Path(geff_path).name}: {exc}")

    if graph is not None:
        print(f"⚠️  estimated_number_of_nodes not found in {Path(geff_path).name} metadata; "
              f"falling back to the sparse GT node count ({graph.number_of_nodes()}).")
        return graph.number_of_nodes()
    return None


# ============================================================
# Node matching: optimal bipartite assignment, <= 7 um, per timepoint
# ============================================================
from scipy.optimize import linear_sum_assignment

MATCH_MAX_DIST_UM = 7.0  # ثابتة حسب شروط المسابقة


def match_nodes(pred_df, gt_df, voxel_size_um, max_dist_um=MATCH_MAX_DIST_UM):
    scale = np.asarray(voxel_size_um)
    matches = {}
    common_ts = sorted(set(pred_df["t"]) & set(gt_df["t"]))
    for t in common_ts:
        p = pred_df[pred_df["t"] == t]
        g = gt_df[gt_df["t"] == t]
        if len(p) == 0 or len(g) == 0:
            continue
        p_um = p[["z", "y", "x"]].to_numpy(dtype=np.float64) * scale
        g_um = g[["z", "y", "x"]].to_numpy(dtype=np.float64) * scale
        d = np.linalg.norm(p_um[:, None, :] - g_um[None, :, :], axis=-1)
        cost = np.where(d <= max_dist_um, d, 1e6)
        row_ind, col_ind = linear_sum_assignment(cost)
        p_ids = p["node_id"].to_numpy()
        g_ids = g["node_id"].to_numpy()
        for r, c in zip(row_ind, col_ind):
            if cost[r, c] < 1e6:
                matches[p_ids[r]] = g_ids[c]
    return matches


# ============================================================
# Edge Jaccard (+ adjusted edge Jaccard)
# ============================================================

def edge_jaccard_metrics(pred_nodes, pred_edges, gt_nodes, gt_edges, voxel_size_um,
                          node_match=None):
    if node_match is None:
        node_match = match_nodes(pred_nodes, gt_nodes, voxel_size_um)

    gt_edge_set = set(zip(gt_edges["source_id"], gt_edges["target_id"]))
    gt_out = gt_edges.groupby("source_id")["target_id"].apply(set).to_dict()
    gt_in = gt_edges.groupby("target_id")["source_id"].apply(set).to_dict()

    tp = 0
    fp = 0
    matched_gt_edges = set()
    for s, tgt in zip(pred_edges["source_id"], pred_edges["target_id"]):
        s_gt = node_match.get(s)
        t_gt = node_match.get(tgt)
        if s_gt is not None and t_gt is not None and (s_gt, t_gt) in gt_edge_set:
            tp += 1
            matched_gt_edges.add((s_gt, t_gt))
            continue
        is_fp = False
        if t_gt is not None and t_gt in gt_in:
            is_fp = True
        if s_gt is not None and s_gt in gt_out:
            is_fp = True
        if is_fp:
            fp += 1

    fn = len(gt_edge_set) - len(matched_gt_edges)
    denom = tp + fp + fn
    jaccard = tp / denom if denom > 0 else 1.0
    return {"tp": tp, "fp": fp, "fn": fn, "jaccard": jaccard, "node_match": node_match}


def adjusted_edge_jaccard(jaccard, t_pred, t_true, a=0.1):
    if not t_true or t_true <= 0:
        return jaccard
    return max(0.0, jaccard * (1.0 - a * (t_pred - t_true) / t_true))


# ============================================================
# Division Jaccard (+/-1 timepoint tolerant fork matching)
# ============================================================
import networkx as nx
from collections import deque


def _descendants_within(children_map, start, max_depth):
    visited = {start}
    frontier = deque([(start, 0)])
    while frontier:
        node, depth = frontier.popleft()
        if depth >= max_depth:
            continue
        for child in children_map.get(node, []):
            if child not in visited:
                visited.add(child)
                frontier.append((child, depth + 1))
    return visited


def division_jaccard_metrics(pred_nodes, pred_edges, gt_nodes, gt_edges, voxel_size_um,
                              node_match=None, lineage_window=4):
    if node_match is None:
        node_match = match_nodes(pred_nodes, gt_nodes, voxel_size_um)

    gt_match_inv = {}
    for p_id, g_id in node_match.items():
        gt_match_inv.setdefault(g_id, set()).add(p_id)

    gt_children = gt_edges.groupby("source_id")["target_id"].apply(list).to_dict()
    gt_parent = dict(zip(gt_edges["target_id"], gt_edges["source_id"]))
    gt_divisions = [n for n, c in gt_children.items() if len(c) >= 2]
    gt_annotated = set(gt_edges["source_id"]) | set(gt_edges["target_id"])

    pred_out_deg = pred_edges.groupby("source_id").size().to_dict()
    pred_dividers = {n for n, c in pred_out_deg.items() if c >= 2}

    pred_graph = nx.DiGraph()
    pred_graph.add_nodes_from(pred_nodes["node_id"])
    pred_graph.add_edges_from(zip(pred_edges["source_id"], pred_edges["target_id"]))
    pred_undirected = pred_graph.to_undirected()

    matched_gt_divisions = set()
    for div in gt_divisions:
        children = gt_children[div][:2]
        if len(children) < 2:
            continue
        pre_split = {div}
        if div in gt_parent:
            pre_split.add(gt_parent[div])
        pre_matched = set().union(*(gt_match_inv.get(n, set()) for n in pre_split)) if pre_split else set()
        lineage_matched = []
        for c in children:
            lineage = _descendants_within(gt_children, c, lineage_window)
            m = set().union(*(gt_match_inv.get(n, set()) for n in lineage)) if lineage else set()
            lineage_matched.append(m)
        if not pre_matched or any(len(m) == 0 for m in lineage_matched):
            continue
        all_matched = pre_matched | lineage_matched[0] | lineage_matched[1]
        all_matched &= set(pred_graph.nodes)
        if len(all_matched) < 2:
            continue
        sub = pred_undirected.subgraph(all_matched)
        if sub.number_of_nodes() == 0 or not nx.is_connected(sub):
            continue
        comp = nx.node_connected_component(pred_undirected, next(iter(all_matched)))
        if not (comp & pred_dividers):
            continue
        matched_gt_divisions.add(div)

    div_tp = len(matched_gt_divisions)
    div_fn = len(gt_divisions) - div_tp
    div_fp = 0
    for pd_node in pred_dividers:
        gt_of = node_match.get(pd_node)
        if gt_of is not None and gt_of in gt_annotated and gt_of not in matched_gt_divisions:
            div_fp += 1

    denom = div_tp + div_fp + div_fn
    jaccard = div_tp / denom if denom > 0 else 1.0
    return {"tp": div_tp, "fp": div_fp, "fn": div_fn, "jaccard": jaccard}


# ============================================================
# Combined local score: micro-averaged, زي "Final score" في metrics.md
# ============================================================

def evaluate_against_train(node_rows_by_dataset, edge_rows_by_dataset,
                            train_dir=TRAIN_DIR, voxel_size_um=VOXEL_SIZE_UM,
                            a=0.1, division_weight=0.1, verbose=True):
    rows = []
    total_dtp = total_dfp = total_dfn = 0
    weighted_adj_sum = weight_sum = 0.0

    for ds_name, pred_nodes in node_rows_by_dataset.items():
        geff_path = Path(train_dir) / f"{ds_name}.geff"
        if not geff_path.exists():
            continue

        graph, metadata = load_geff_graph(geff_path)
        gt_nodes = geff_nodes_df(graph)
        gt_edges = geff_edges_df(graph)
        pred_edges = edge_rows_by_dataset[ds_name]

        node_match = match_nodes(pred_nodes, gt_nodes, voxel_size_um)
        em = edge_jaccard_metrics(pred_nodes, pred_edges, gt_nodes, gt_edges,
                                   voxel_size_um, node_match=node_match)
        dm = division_jaccard_metrics(pred_nodes, pred_edges, gt_nodes, gt_edges,
                                       voxel_size_um, node_match=node_match)

        t_true = estimated_true_node_count(geff_path, graph, metadata)
        t_pred = len(pred_nodes)
        adj = adjusted_edge_jaccard(em["jaccard"], t_pred, t_true, a=a)

        w_i = em["tp"] + em["fp"] + em["fn"]
        weighted_adj_sum += adj * w_i
        weight_sum += w_i
        total_dtp += dm["tp"]; total_dfp += dm["fp"]; total_dfn += dm["fn"]

        rows.append({
            "dataset": ds_name, "t_pred": t_pred, "t_true": t_true,
            "edge_tp": em["tp"], "edge_fp": em["fp"], "edge_fn": em["fn"],
            "edge_jaccard": round(em["jaccard"], 4),
            "adjusted_edge_jaccard": round(adj, 4),
            "div_tp": dm["tp"], "div_fp": dm["fp"], "div_fn": dm["fn"],
            "div_jaccard": round(dm["jaccard"], 4),
        })

    results_df = pd.DataFrame(rows)
    micro_adjusted = weighted_adj_sum / weight_sum if weight_sum > 0 else 0.0
    div_denom = total_dtp + total_dfp + total_dfn
    micro_div = total_dtp / div_denom if div_denom > 0 else 0.0
    final_score = micro_adjusted + division_weight * micro_div

    if verbose:
        print("=" * 70)
        display(results_df)
        print("-" * 70)
        print(f"Micro-averaged adjusted edge Jaccard : {micro_adjusted:.4f}")
        print(f"Micro-averaged division Jaccard       : {micro_div:.4f}")
        print(f"Final score (edge + {division_weight}*division)  : {final_score:.4f}")
        print("=" * 70)

    return results_df, final_score

In [15]:
# ============================================================
# شغّل الـ pipeline على عدد من TRAIN datasets واحسب السكور محليًا
# (بيستخدم process_dataset() و hoct_model الموجودين فوق في النوتبوك)
# ============================================================
MAX_TRAIN_DATASETS_FOR_VALIDATION = 3  # ابدأ بعدد صغير — كل dataset ممكن ياخد دقايق

try:
    import geff  # noqa: F401
    _GEFF_OK = True
except ImportError:
    _GEFF_OK = False

if _GEFF_OK:
    train_zarrs = sorted(TRAIN_DIR.glob("*.zarr"))
    train_zarrs = [p for p in train_zarrs if (TRAIN_DIR / f"{p.stem}.geff").exists()]
    train_zarrs = train_zarrs[:MAX_TRAIN_DATASETS_FOR_VALIDATION]

    val_node_rows, val_edge_rows = {}, {}
    for ds_path in train_zarrs:
        n_rows, e_rows = process_dataset(ds_path, hoct_model, verbose=True)
        val_node_rows[ds_path.stem] = n_rows
        val_edge_rows[ds_path.stem] = e_rows

    if val_node_rows:
        val_results_df, val_final_score = evaluate_against_train(val_node_rows, val_edge_rows)
    else:
        print("مفيش train datasets ليها .geff مطابق تحت", TRAIN_DIR)
else:
    print("مقدرش أعمل التقييم — geff مش متاحة.")

Processing dataset: 44b6_0113de3b  (T=100, shape=(64, 256, 256))


44b6_0113de3b: frames:   0%|          | 0/100 [00:00<?, ?it/s]

Detected 42980 nuclei across 100 frames (429.8 / frame on average)
Running HOCT tracking ...


Adding edges:   0%|          | 0/100 [00:00<?, ?it/s]

[09/27/26 20:23:14] WARNING  No nodes found for time point -3                                ]8;id=883010;file:///usr/local/lib/python3.12/dist-packages/tracksdata/edges/_distance_edges.py\_distance_edges.py]8;;\:]8;id=292011;file:///usr/local/lib/python3.12/dist-packages/tracksdata/edges/_distance_edges.py#295\295]8;;\

...

[09/27/26 21:00:18] WARNING  Solver failed with Gurobi, trying Scip.                             ]8;id=77468;file:///usr/local/lib/python3.12/dist-packages/tracksdata/solvers/_ilp_solver.py\_ilp_solver.py]8;;\:]8;id=354085;file:///usr/local/lib/python3.12/dist-packages/tracksdata/solvers/_ilp_solver.py#363\363]8;;\
                             Got error:                                                                            
                             Gurobi license is not available.                                                      

[09/27/26 21:03:33] WARNING  Solver failed with Gurobi, trying Scip.                             ]8;id=840048;file:///usr/local/lib/python3.12/dist-packages/tracksdata/solvers/_ilp_solver.py\_ilp_solver.py]8;;\:]8;id=912317;file:///usr/local/lib/python3.12/dist-packages/tracksdata/solvers/_ilp_solver.py#363\363]8;;\
                             Got error:                                                                            
                             Gurobi license is not available.                                                      

44b6_0c582fdc: 50186 node rows, 48148 edge rows


,dataset,t_pred,t_true,edge_tp,edge_fp,edge_fn,edge_jaccard,adjusted_edge_jaccard,div_tp,div_fp,div_fn,div_jaccard
0,44b6_0113de3b,42977,25755,33,30,17,0.4125,0.3849,0,4,0,0.0
1,44b6_0b24845f,83475,32795,26,31,23,0.3250,0.2748,0,1,0,0.0
2,44b6_0c582fdc,50186,27958,49,37,21,0.4579,0.4215,0,4,0,0.0


----------------------------------------------------------------------
Micro-averaged adjusted edge Jaccard : 0.3666
Micro-averaged division Jaccard       : 0.0000
Final score (edge + 0.1*division)  : 0.3666


In [16]:
# ============================================================
# Sweep: جرّب أكتر من عتبة كشف، واحسب T_pred/T_true والسكور لكل واحدة
# ============================================================

def sweep_detection_threshold(dataset_paths, hoct_model,
                               log_threshold_fractions,
                               mask_threshold_fractions,
                               voxel_size_um=VOXEL_SIZE_UM):
    sweep_rows = []
    for log_thr in log_threshold_fractions:
        for mask_thr in mask_threshold_fractions:
            node_rows, edge_rows = {}, {}
            for ds_path in dataset_paths:
                n_rows, e_rows = process_dataset(
                    ds_path, hoct_model,
                    log_threshold_fraction=log_thr,
                    mask_threshold_fraction=mask_thr,
                    verbose=False,
                )
                node_rows[ds_path.stem] = n_rows
                edge_rows[ds_path.stem] = e_rows

            results_df, final_score = evaluate_against_train(
                node_rows, edge_rows, voxel_size_um=voxel_size_um, verbose=False,
            )
            if len(results_df) == 0:
                continue
            ratio = (results_df["t_pred"] / results_df["t_true"]).mean()
            sweep_rows.append({
                "log_threshold_fraction": log_thr,
                "mask_threshold_fraction": mask_thr,
                "mean_T_pred_over_T_true": round(ratio, 3),
                "final_score": round(final_score, 4),
            })
            print(f"log_thr={log_thr:<6} mask_thr={mask_thr:<6} "
                  f"T_pred/T_true={ratio:.3f}  score={final_score:.4f}")

    sweep_df = pd.DataFrame(sweep_rows).sort_values("final_score", ascending=False)
    return sweep_df

In [17]:
# ============================================================
# جرّب عتبات أعلى من الحالية (log=0.03, mask=0.08) عشان نقلل الكشف الزايد
# ابدأ بـ dataset واحد بس (أسرع)؛ لما تلاقي إعداد كويس، جرّبه على التلاتة
# ============================================================
sweep_df = sweep_detection_threshold(
    dataset_paths=train_zarrs[:1],          # dataset واحد بس أول حاجة
    hoct_model=hoct_model,
    log_threshold_fractions=(0.05, 0.08, 0.12),
    mask_threshold_fractions=(0.10, 0.14),
)
display(sweep_df)

Adding edges:   0%|          | 0/100 [00:00<?, ?it/s]

[09/27/26 21:25:12] WARNING  No nodes found for time point -3                                ]8;id=684780;file:///usr/local/lib/python3.12/dist-packages/tracksdata/edges/_distance_edges.py\_distance_edges.py]8;;\:]8;id=966424;file:///usr/local/lib/python3.12/dist-packages/tracksdata/edges/_distance_edges.py#295\295]8;;\

                    WARNING  No nodes found for time point -2                                ]8;id=979583;file:///usr/local/lib/python3.12/dist-packages/tracksdata/edges/_distance_edges.py\_distance_edges.py]8;;\:]8;id=401409;file:///usr/local/lib/python3.12/dist-packages/tracksdata/edges/_distance_edges.py#295\295]8;;\

                    WARNING  No nodes found for time point -1                                ]8;id=323878;file:///usr/local/lib/python3.12/dist-packages/tracksdata/edges/_distance_edges.py\_distance_edges.py]8;;\:]8;id=111290;file:///usr/local/lib/python3.12/dist-packages/tracksdata/edges/_distance_edges.py#295\295]8;;\

                    WARNING  No valid edges found for time point 0                           ]8;id=506439;file:///usr/local/lib/python3.12/dist-packages/tracksdata/edges/_distance_edges.py\_distance_edges.py]8;;\:]8;id=33807;file:///usr/local/lib/python3.12/dist-packages/tracksdata/edges/_distance_edges.py#337\337]8;;\

...

Adding node attributes: 100%|██████████| 100/100 [00:00<00:00, 1346.31it/s]
[W927 22:29:48.346607912 init.cpp:862] Warning: Use _jit_set_fusion_strategy, bailout depth is deprecated. Setting to (STATIC, 0) (function operator())
Model inference: 544it [00:18, 29.48it/s]


[09/27/26 22:30:33] WARNING  Solver failed with Gurobi, trying Scip.                             ]8;id=446807;file:///usr/local/lib/python3.12/dist-packages/tracksdata/solvers/_ilp_solver.py\_ilp_solver.py]8;;\:]8;id=501423;file:///usr/local/lib/python3.12/dist-packages/tracksdata/solvers/_ilp_solver.py#363\363]8;;\
                             Got error:                                                                            
                             Gurobi license is not available.                                                      

[09/27/26 22:32:59] WARNING  Solver failed with Gurobi, trying Scip.                             ]8;id=43473;file:///usr/local/lib/python3.12/dist-packages/tracksdata/solvers/_ilp_solver.py\_ilp_solver.py]8;;\:]8;id=657615;file:///usr/local/lib/python3.12/dist-packages/tracksdata/solvers/_ilp_solver.py#363\363]8;;\
                             Got error:                                                                            
                             Gurobi license is not available.                                                      

log_thr=0.12   mask_thr=0.14   T_pred/T_true=1.667  score=0.4541


,log_threshold_fraction,mask_threshold_fraction,mean_T_pred_over_T_true,final_score
1,0.05,0.14,1.667,0.4541
3,0.08,0.14,1.667,0.4541
5,0.12,0.14,1.667,0.4541
0,0.05,0.10,1.669,0.3528
2,0.08,0.10,1.669,0.3528
4,0.12,0.10,1.669,0.3528
